# 03. Reasoning Effort & Graceful Fallback

This notebook (**AI-103 -> Section 01**) sends a diagnostic/reasoning-heavy prompt with the `reasoning={"effort": "high"}` parameter, and wraps the call in a `try`/`except` that falls back to a plain call if the deployed model doesn't support the `reasoning` field. It's a small but realistic pattern for writing code that works across heterogeneous Azure OpenAI deployments.

**Difficulty: Intermediate**


## Prerequisites

**Python packages** (already in this repo's root `requirements.txt`):
- `openai` (this notebook also imports `BadRequestError` from it)
- `azure-identity`
- `python-dotenv`

**Azure resources**
- For the *full* effect (the `reasoning.effort` branch actually taking hold) you need a reasoning-capable deployment (an o-series or GPT-5-class reasoning deployment). Against a non-reasoning deployment like `gpt-4.1` the code still runs correctly -- it just always takes the `except` fallback branch.
- Entra ID auth via `az login`, same as notebook 02.

**Environment variables** -- add to the root `.env`:
```
AZURE_OPENAI_ENDPOINT=https://<your-resource>.services.ai.azure.com/openai/v1
AZURE_OPENAI_DEPLOYMENT=gpt-4.1
```


## What You'll Learn
- The `reasoning={"effort": ...}` parameter for reasoning-capable models
- Catching `openai.BadRequestError` and inspecting its message to decide how to recover
- A defensive-coding pattern for handling capability differences across model deployments
- The cost/latency trade-off that reasoning effort levels represent


### 1. Imports and configuration

Same Entra ID setup as the previous notebook, plus `BadRequestError` imported explicitly so we can catch it by type rather than a bare `except Exception`.


In [ ]:
import os

from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from dotenv import load_dotenv
from openai import BadRequestError, OpenAI

load_dotenv()

endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
deployment_name = os.getenv("AZURE_OPENAI_DEPLOYMENT", "gpt-4.1")
token_provider = get_bearer_token_provider(DefaultAzureCredential(), "https://ai.azure.com/.default")


### 2. Build the client


In [ ]:
client = OpenAI(
    base_url=endpoint,
    api_key=token_provider,
)


### 3. Define the problem prompt

A deliberately reasoning-heavy prompt (diagnosing an intermittent concurrency bug) -- the kind of task where a reasoning model's extra internal deliberation tends to produce a better answer than a single fast forward-pass would.


In [ ]:
problem = """
A distributed e-commerce system is experiencing intermittent checkout failures 
during peak traffic. The failures appear random, affect roughly 3 percent of the transactions, 
and only occur when inventory checks and payment processing run concurrently. 
Identify the most likely root cause and propose a solution.
"""


### 4. Call with reasoning effort, falling back if unsupported

`reasoning={"effort": "high"}` asks a reasoning-capable model to spend more internal reasoning tokens before answering. Not every deployed model accepts this field -- calling it against a non-reasoning deployment raises `openai.BadRequestError`. The `except` clause checks whether the error specifically mentions `reasoning.effort`; if so, it retries the identical call without that parameter. Any other kind of `BadRequestError` is re-raised rather than silently swallowed.

**Exam tip:** AI-102 tests whether you understand that Azure OpenAI model families differ in which parameters they accept (reasoning effort, structured outputs, function calling, etc.). Writing code that probes for and gracefully degrades around capability mismatches -- instead of hardcoding assumptions about one specific deployment -- is a resilience pattern the exam associates with production-grade agent apps. Also: reasoning effort levels trade off latency and (billed) reasoning-token cost against answer quality/depth.

**Alternatives:** Instead of a runtime try/except probe, you could look up a deployment's supported capabilities upfront (e.g. from your own configuration or the AI Foundry Model Catalog metadata) and branch before calling -- trading a bit of setup complexity for one fewer network round trip on the unsupported path.


In [ ]:
try:
    response = client.responses.create(
        model=deployment_name,
        instructions="You are a senior software architect.",
        input=problem,
        reasoning={"effort": "high"},
    )
except BadRequestError as exc:
    if "reasoning.effort" in str(exc):
        response = client.responses.create(
            model=deployment_name,
            instructions="You are a senior software architect.",
            input=problem,
        )
    else:
        raise


### 5. Read the answer


In [ ]:
print(f"answer: {response.output_text}")


## Summary

You sent a reasoning-heavy prompt with `reasoning={"effort": "high"}`, and the code degrades gracefully to a plain call if the deployment doesn't support that parameter -- so the same notebook runs correctly against both reasoning and non-reasoning deployments.


## Try It Yourself
1. **Easy:** Swap `problem` for a different troubleshooting scenario of your own.
2. **Medium:** If you have a reasoning-capable deployment, try `"effort": "low"` and `"effort": "medium"` and compare answer depth/length against `"high"`.
3. **Advanced:** Wrap the whole call in a small retry-with-backoff helper that also handles `openai.RateLimitError`, so the reasoning fallback and rate-limit retry compose cleanly instead of being separate ad hoc `try`/`except` blocks.
